In [16]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df_touchpoints = pd.read_csv("touchpoints_clean.csv")

In [17]:
# FIRST TOUCH

converted_ids = df_touchpoints[df_touchpoints['converted'] == 1]['customer_id'].unique()
df_converted = df_touchpoints[df_touchpoints['customer_id'].isin(converted_ids)]

first_touch_attribution = (
    df_converted[df_converted['position'] == 1]
    .groupby('channel')
    .size()
    .reset_index(name='conversions')
    .sort_values('conversions', ascending=False)
)
first_touch_attribution['%'] = (
    first_touch_attribution['conversions'] / first_touch_attribution['conversions'].sum() * 100
).round(1)

first_touch_attribution

,channel,conversions,%
1,display,44804,30.0
4,social,44672,30.0
0,affiliate,44487,29.8
2,email,7612,5.1
3,retargeting,7575,5.1


In [18]:
# répartition budget first touch
first_channel = (
    df_converted[df_converted['position'] == 1][['customer_id', 'channel']]
    .rename(columns={'channel': 'first_channel'})
)

cost_per_customer = df_converted.groupby('customer_id')['cost'].sum().reset_index()

df_budget = cost_per_customer.merge(first_channel, on='customer_id')

budget_first_touch = (
    df_budget.groupby('first_channel')['cost']
    .sum()
    .reset_index()
    .rename(columns={'first_channel': 'channel', 'cost': 'budget'})
    .sort_values('budget', ascending=False)
    .reset_index(drop=True)
)

budget_first_touch['budget'] = budget_first_touch['budget'].round(2)
budget_first_touch['%'] = (
    budget_first_touch['budget'] / budget_first_touch['budget'].sum() * 100
).round(1)

budget_first_touch

,channel,budget,%
0,display,2944164.83,32.3
1,affiliate,2935242.72,32.2
2,social,2921621.95,32.1
3,retargeting,160888.14,1.8
4,email,149597.36,1.6


In [19]:
# LAST TOUCH

df_converted_last_touch = df_touchpoints[df_touchpoints['customer_id'].isin(converted_ids)]

last_touch_attribution = (
    df_converted_last_touch[df_converted_last_touch['position'] == df_converted_last_touch['position'].max()]
    .groupby('channel')
    .size()
    .reset_index(name='conversions')
    .sort_values('conversions', ascending=False)
)
last_touch_attribution['%'] = (
    last_touch_attribution['conversions'] / last_touch_attribution['conversions'].sum() * 100
).round(1)

last_touch_attribution

,channel,conversions,%
2,retargeting,3108,25.6
0,direct,3084,25.4
3,search_paid,2980,24.5
1,email,2976,24.5


In [20]:
# répartition budget last_touch
last_channel = (
    df_converted_last_touch.loc[df_converted_last_touch.groupby('customer_id')['position'].idxmax(), ['customer_id', 'channel']]
    .rename(columns={'channel': 'last_channel'})
)

cost_per_customer = df_converted_last_touch.groupby('customer_id')['cost'].sum().reset_index()

df_budget_last = cost_per_customer.merge(last_channel, on='customer_id')

budget_last_touch = (
    df_budget_last.groupby('last_channel')['cost']
    .sum()
    .reset_index()
    .rename(columns={'last_channel': 'channel', 'cost': 'budget'})
    .sort_values('budget', ascending=False)
    .reset_index(drop=True)
)

budget_last_touch['budget'] = budget_last_touch['budget'].round(2)
budget_last_touch['%'] = (
    budget_last_touch['budget'] / budget_last_touch['budget'].sum() * 100
).round(1)

budget_last_touch

,channel,budget,%
0,retargeting,128190.45,25.6
1,search_paid,120857.56,24.2
2,email,119933.03,24.0
3,direct,115461.95,23.1
4,affiliate,5601.05,1.1
5,social,5203.39,1.0
6,display,4584.99,0.9


In [21]:
# LINEAIRE

df_converted_linear = df_touchpoints[df_touchpoints['customer_id'].isin(converted_ids)].copy()

df_converted_linear['credit'] = df_converted_linear.groupby('customer_id')['channel'].transform(lambda x: 1 / len(x))

linear_attribution = (
    df_converted_linear.groupby('channel')['credit']
    .sum()
    .reset_index(name='conversions')
    .sort_values('conversions', ascending=False)
)
linear_attribution['%'] = (
    linear_attribution['conversions'] / linear_attribution['conversions'].sum() * 100
).round(1)

linear_attribution

,channel,conversions,%
2,display,4887.818833,17.6
0,affiliate,4847.514096,17.5
6,social,4846.290794,17.5
3,email,3850.989440,13.9
4,retargeting,3821.402107,13.8
1,direct,2761.845363,10.0
5,search_paid,2737.139368,9.9


In [25]:
# répartition budget linéaire

cost_per_customer = df_converted_linear.groupby('customer_id')['cost'].sum().reset_index()

df_linear_budget = (
    df_converted_linear[['customer_id', 'channel', 'credit']]
    .merge(cost_per_customer, on='customer_id')
)
df_linear_budget['budget_attributed'] = df_linear_budget['credit'] * df_linear_budget['cost']

budget_linear = (
    df_linear_budget.groupby('channel')['budget_attributed']
    .sum()
    .reset_index()
    .rename(columns={'budget_attributed': 'budget'})
    .sort_values('budget', ascending=False)
    .reset_index(drop=True)
)

budget_linear['budget'] = budget_linear['budget'].round(2)
budget_linear['%'] = (
    budget_linear['budget'] / budget_linear['budget'].sum() * 100
).round(1)

budget_linear

,channel,budget,%
0,affiliate,87972.82,17.6
1,social,82238.22,16.5
2,display,80813.58,16.2
3,retargeting,70088.49,14.0
4,email,66556.54,13.3
5,search_paid,57608.42,11.5
6,direct,54554.37,10.9


In [27]:
# MODEL U

df_converted_u = df_touchpoints[df_touchpoints['customer_id'].isin(converted_ids)].copy()

def u_shape_credit(group):
    n = len(group)
    group = group.sort_values('position')
    credits = pd.Series(0.0, index=group.index)

    if n == 1:
        credits.iloc[0] = 1.0
    elif n == 2:
        credits.iloc[0] = 0.5
        credits.iloc[1] = 0.5
    else:
        credits.iloc[0] = 0.4              
        credits.iloc[-1] = 0.4             
        credits.iloc[1:-1] = 0.2 / (n - 2) 

    return credits

df_converted_u['credit'] = (
    df_converted_u.groupby('customer_id', group_keys=False)
    .apply(u_shape_credit, include_groups=False)
)

u_shape_attribution = (
    df_converted_u.groupby('channel')['credit']
    .sum()
    .reset_index(name='conversions')
    .sort_values('conversions', ascending=False)
)
u_shape_attribution['%'] = (
    u_shape_attribution['conversions'] / u_shape_attribution['conversions'].sum() * 100
).round(1)

u_shape_attribution

,channel,conversions,%
2,display,4540.079849,16.4
6,social,4508.623860,16.2
0,affiliate,4452.207794,16.0
3,email,4104.053301,14.8
4,retargeting,4082.599215,14.7
1,direct,3033.525804,10.9
5,search_paid,3031.910177,10.9


In [28]:
# répartition budget model U

cost_per_customer = df_converted_u.groupby('customer_id')['cost'].sum().reset_index()

df_u_budget = (
    df_converted_u[['customer_id', 'channel', 'credit']]
    .merge(cost_per_customer, on='customer_id')
)
df_u_budget['budget_attributed'] = df_u_budget['credit'] * df_u_budget['cost']

df_u_budget = (
    df_u_budget.groupby('channel')['budget_attributed']
    .sum()
    .reset_index()
    .rename(columns={'budget_attributed': 'budget'})
    .sort_values('budget', ascending=False)
    .reset_index(drop=True)
)

df_u_budget['budget'] = df_u_budget['budget'].round(2)
df_u_budget['%'] = (
    df_u_budget['budget'] / df_u_budget['budget'].sum() * 100
).round(1)

df_u_budget

,channel,budget,%
0,affiliate,83008.83,16.6
1,social,78508.24,15.7
2,display,77589.93,15.5
3,retargeting,72842.36,14.6
4,email,70626.34,14.1
5,search_paid,59999.53,12.0
6,direct,57257.21,11.5
